In [2]:
from googleapiclient.discovery import build
from google.oauth2 import service_account

SCOPES = ['https://www.googleapis.com/auth/drive.readonly']

creds = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES
)
drive_service = build('drive', 'v3', credentials=creds)
folder_id = '1JfFPowlXj4dQIZCl7H_rI8XlmrsjDCjqJbT8-F-pQNIQutQsMv3m2YNPxHW6XNlaZF9VmzKK'
query = f"'{folder_id}' in parents"
results = drive_service.files().list(q=query, fields="files(id, name)").execute()
files = results.get('files', [])
if not files:
    print('No files found in the folder.')
else:
    print(f"Found {len(files)} file(s) in the folder:")
    for file in files:
        print(f"{file['name']} (ID: {file['id']})")

Found 5 file(s) in the folder:
Rithul_Rakesh.pdf (ID: 1huTInCSx2czjeoTzFYccB328nZpGdrFm)
Raghav Balakrishnan_PES University.pdf (ID: 1FthTjAvhCT-irqmdo85HZTMsRW4GrxQO)
nitheesh_resume.pdf (ID: 1f6R3U8Gn4A9bpgaooXDKdd3S2bCzuB-q)
NEVILLE_JOSEPH.pdf (ID: 1irDGX6ULMyU3qS6OqE7LSJallnzUuNw2)
Sreecharan_pes_updated (3).pdf (ID: 1JFi9nQ88MA2pMLv1OQ3mtnIpCGmqgl4C)


In [3]:
from googleapiclient.discovery import build
from google.oauth2 import service_account
import os
from googleapiclient.http import MediaIoBaseDownload

# Define the output directory
output_directory_path = "C:\\Users\sreec\PycharmProjects\ENTERPRISE_RAG\GENAI_PROJECT\GoogleDriveFiles"  # Change this path as needed
# Ensure the directory exists
os.makedirs(output_directory_path, exist_ok=True)

SCOPES = ['https://www.googleapis.com/auth/drive']

creds = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES
)
drive_service = build('drive', 'v3', credentials=creds)

folder_id = '1JfFPowlXj4dQIZCl7H_rI8XlmrsjDCjqJbT8-F-pQNIQutQsMv3m2YNPxHW6XNlaZF9VmzKK'
query = f"'{folder_id}' in parents and trashed=false"
results = drive_service.files().list(q=query, fields="files(id, name)").execute()
files = results.get('files', [])

if not files:
    print('No files found in the folder.')
else:
    print(f"Found {len(files)} file(s) in the folder:")    
    for file in files:
        file_id = file['id']
        file_name = file['name']
        
        print(f"Downloading: {file_name} (ID: {file_id})")

        # Request to download file
        request = drive_service.files().get_media(fileId=file_id)
        file_path = os.path.join(output_directory_path, file_name)  # Save in the specified directory

        # Download the file
        with open(file_path, 'wb') as f:
            downloader = MediaIoBaseDownload(f, request)
            done = False
            while not done:
                status, done = downloader.next_chunk()
                print(f"Download {int(status.progress() * 100)}% complete")

        print(f"File downloaded: {file_path}")

Found 5 file(s) in the folder:
Downloading: Rithul_Rakesh.pdf (ID: 1huTInCSx2czjeoTzFYccB328nZpGdrFm)
Download 100% complete
File downloaded: C:\Users\sreec\PycharmProjects\ENTERPRISE_RAG\GENAI_PROJECT\GoogleDriveFiles\Rithul_Rakesh.pdf
Downloading: Raghav Balakrishnan_PES University.pdf (ID: 1FthTjAvhCT-irqmdo85HZTMsRW4GrxQO)
Download 100% complete
File downloaded: C:\Users\sreec\PycharmProjects\ENTERPRISE_RAG\GENAI_PROJECT\GoogleDriveFiles\Raghav Balakrishnan_PES University.pdf
Downloading: nitheesh_resume.pdf (ID: 1f6R3U8Gn4A9bpgaooXDKdd3S2bCzuB-q)
Download 100% complete
File downloaded: C:\Users\sreec\PycharmProjects\ENTERPRISE_RAG\GENAI_PROJECT\GoogleDriveFiles\nitheesh_resume.pdf
Downloading: NEVILLE_JOSEPH.pdf (ID: 1irDGX6ULMyU3qS6OqE7LSJallnzUuNw2)
Download 100% complete
File downloaded: C:\Users\sreec\PycharmProjects\ENTERPRISE_RAG\GENAI_PROJECT\GoogleDriveFiles\NEVILLE_JOSEPH.pdf
Downloading: Sreecharan_pes_updated (3).pdf (ID: 1JFi9nQ88MA2pMLv1OQ3mtnIpCGmqgl4C)
Download 100%

In [4]:
import os
import nest_asyncio
nest_asyncio.apply()

In [5]:
import sys
sys.path.append(r"C:\Users\sreec\PycharmProjects\ENTERPRISE_RAG")
from Path_Of_GoogleDriveFiles import get_list_paths
output_directory_list = get_list_paths(output_directory_path)
output_directory_list

['C:\\Users\\sreec\\PycharmProjects\\ENTERPRISE_RAG\\GENAI_PROJECT\\GoogleDriveFiles\\NEVILLE_JOSEPH.pdf',
 'C:\\Users\\sreec\\PycharmProjects\\ENTERPRISE_RAG\\GENAI_PROJECT\\GoogleDriveFiles\\nitheesh_resume.pdf',
 'C:\\Users\\sreec\\PycharmProjects\\ENTERPRISE_RAG\\GENAI_PROJECT\\GoogleDriveFiles\\Raghav Balakrishnan_PES University.pdf',
 'C:\\Users\\sreec\\PycharmProjects\\ENTERPRISE_RAG\\GENAI_PROJECT\\GoogleDriveFiles\\Rithul_Rakesh.pdf',
 'C:\\Users\\sreec\\PycharmProjects\\ENTERPRISE_RAG\\GENAI_PROJECT\\GoogleDriveFiles\\Sreecharan_pes_updated (3).pdf']

In [7]:
storage_text = {}

In [8]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="mixtral-8x7b-32768",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

In [9]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are an assistant that normalizes the given JSON input into the following structured format:
            and do not randomly insert 
            {{
              "contact_information": {{
                "name": "<Full Name>",
                "contact": {{
                  "phone": "<Phone Number>",
                  "email": "<Email Address>",
                  "github": "<GitHub Profile URL>",
                  "linkedin": "<LinkedIn Profile URL>",
                  "website": "<Personal Website or Portfolio URL>"
                }}
              }},
              "education": [
                {{
                  "institution": "<University or College Name>",
                  "degree": "<Degree Name>",
                  "field_of_study": "<Field of Study>",
                  "duration": "<Start Year - End Year or Present>",
                  "location": "<City, Country>"
                }}
              ],
              "experience": [
                {{
                  "company": "<Company Name>",
                  "position": "<Job Title>",
                  "duration": "<Start Date - End Date or Present>",
                  "location": "<City, Country>",
                  "responsibilities": [
                    "<Responsibility or achievement 1>",
                    "<Responsibility or achievement 2>",
                    "<Responsibility or achievement 3>"
                  ]
                }}
              ],
              "technical_skills": {{
                "languages": ["<Programming Language 1>", "<Programming Language 2>", "..."],
                "technologies": ["<Technology 1>", "<Technology 2>", "..."]
              }},
              "projects": [
                {{
                  "name": "<Project Name>",
                  "description": "<Brief description of the project highlighting its functionality and purpose>"
                }}
              ],
              "courses_taught": [
                {{
                  "title": "<Course Name>",
                  "duration": "<Start Date - End Date>",
                  "description": "<Brief description of the course and topics covered>"
                }}
              ],
              "clubs_and_community": [
                {{
                  "name": "<Club or Organization Name>",
                  "role": "<Role in the Organization>",
                  "description": "<Contributions and activities>"
                }}
              ],
              "achievements": [
                {{
                  "event": "<Event or Competition Name>",
                  "placement": "<Rank or Award>",
                  "description": "<Brief details on the competition and accomplishment>"
                }}
              ],
              "publications": [
              {{
                "title": "<Publication Title>",
                "authors": ["<Author 1>", "<Author 2>", "..."],
                "journal_or_conference": "<Journal or Conference Name>",
                "year": "<Year of Publication>",
                "doi": "<DOI or URL>",
                "abstract": "<Brief summary of the publication>"
              }}
              ]
            }}
            and just output the json, not anything else strictly, and follow the format mentioned strictly"""
        ),
        ("human", "{input}")  # Correct placeholder syntax
    ]
)

In [10]:
import json
import re
from llama_parse import LlamaParse
from langchain.chains import LLMChain

def get_education_profile(file_path, llm, prompt):
    """
    Parses a resume file into a structured JSON format using LlamaParse and a language model.

    Args:
        file_path (str): Path to the resume file.
        llm: Initialized language model.
        prompt: Initialized prompt template.

    Returns:
        tuple: (parsed_json_data, raw_text) or (None, raw_text) if parsing fails.
    """
    # Initialize the chain
    chain = llm | prompt

    # Parse the file with LlamaParse
    try:
        parsed_data = LlamaParse(result_type="markdown").load_data(file_path)
        raw_text = parsed_data[0].text if parsed_data else ""
    except Exception as e:
        print(f"Error parsing file {file_path} with LlamaParse: {e}")
        return None, None

    if not raw_text:
        print(f"Error: No data extracted from the file {file_path}.")
        return None, None

    # Prepare input for the LLM chain
    input_data = {"input": raw_text}

    try:
        # Invoke the LLM chain to normalize the JSON
        response = chain.invoke(input_data)
        response_content = response.get("text", "") if isinstance(response, dict) else response

        # Extract JSON content from the response
        match = re.search(r'```json\n(.*?)\n```', response_content, re.DOTALL)
        cleaned_json_string = match.group(1).strip() if match else response_content.strip()

        if not cleaned_json_string:
            print(f"Error: Received empty JSON response for file {file_path}.")
            return None, raw_text

        # Normalize JSON keys by replacing escaped underscores
        cleaned_json_string = cleaned_json_string.replace("\\_", "_")

        # Parse the cleaned JSON string
        json_data = json.loads(cleaned_json_string)
        return json_data, raw_text

    except json.JSONDecodeError as e:
        print(f"Error Parsing JSON for file {file_path}: {e}")
        print("Raw JSON String:", cleaned_json_string)  # Debugging
        return None, raw_text
    except Exception as e:
        print(f"Error processing file {file_path}: {e}")
        return None, raw_text

In [11]:
import json
import re
from llama_parse import LlamaParse
from langchain_core.prompts import ChatPromptTemplate

def get_education_profile(file_path):
    parsing_instruction = """
    Parse Me this Resume in the form of a json and follow the given format strictly
                {{
              "contact_information": {{
                "name": "<Full Name>",
                "contact": {{
                  "phone": "<Phone Number>",
                  "email": "<Email Address>",
                  "github": "<GitHub Profile URL>",
                  "linkedin": "<LinkedIn Profile URL>",
                  "website": "<Personal Website or Portfolio URL>"
                }}
              }},
              "education": [
                {{
                  "institution": "<University or College Name>",
                  "degree": "<Degree Name>",
                  "field_of_study": "<Field of Study>",
                  "duration": "<Start Year - End Year or Present>",
                  "location": "<City, Country>"
                }}
              ],
              "experience": [
                {{
                  "company": "<Company Name>",
                  "position": "<Job Title>",
                  "duration": "<Start Date - End Date or Present>",
                  "location": "<City, Country>",
                  "responsibilities": [
                    "<Responsibility or achievement 1>",
                    "<Responsibility or achievement 2>",
                    "<Responsibility or achievement 3>"
                  ]
                }}
              ],
              "technical_skills": {{
                "languages": ["<Programming Language 1>", "<Programming Language 2>", "..."],
                "technologies": ["<Technology 1>", "<Technology 2>", "..."]
              }},
              "projects": [
                {{
                  "name": "<Project Name>",
                  "description": "<Brief description of the project highlighting its functionality and purpose>"
                }}
              ],
              "courses_taught": [
                {{
                  "title": "<Course Name>",
                  "duration": "<Start Date - End Date>",
                  "description": "<Brief description of the course and topics covered>"
                }}
              ],
              "clubs_and_community": [
                {{
                  "name": "<Club or Organization Name>",
                  "role": "<Role in the Organization>",
                  "description": "<Contributions and activities>"
                }}
              ],
              "achievements": [
                {{
                  "event": "<Event or Competition Name>",
                  "placement": "<Rank or Award>",
                  "description": "<Brief details on the competition and accomplishment>"
                }}
              ],
              "publications": [
                {{
                "title": "<Publication Title>",
                "authors": ["<Author 1>", "<Author 2>", "..."],
                "journal_or_conference": "<Journal or Conference Name>",
                "year": "<Year of Publication>",
                "doi": "<DOI or URL>",
                "abstract": "<Brief summary of the publication>"
                }}
                ]
            }}
    """

    chain = prompt | llm  # Ensure `llm` is defined correctly

    # Parse the file with LlamaParse
    parsed_data = LlamaParse(result_type="markdown", parsing_instruction=parsing_instruction).load_data(file_path)
    raw_text = parsed_data[0].text if parsed_data else ""

    if not raw_text:
        print("Error: No data extracted from the file.")
        return None, None

    input_data = {
        "input_type": "Json format in the form of string",
        "output_type": "Json format in the form of string",
        "input": raw_text
    }
    response = chain.invoke(input_data)

    if not response or not response.content:
        print("Error: No response from chain")
        return None, raw_text

    # Extract JSON content from the response
    match = re.search(r'```json\n(.*?)\n```', response.content, re.DOTALL)
    cleaned_json_string = match.group(1).strip() if match else response.content.strip()
    print(cleaned_json_string)
    if not cleaned_json_string:
        print("Error: Received empty JSON response")
        return None, raw_text

    try:
        json_data = json.loads(cleaned_json_string)
        return json_data, raw_text
    except json.JSONDecodeError as e:
        print("Error Parsing JSON:", e)
        print("Raw JSON String:", cleaned_json_string)  # Debugging
        return None, raw_text

In [12]:
def aggMetadata(directory):
    storage = {}
    storage_text = {}
    missing_contact_info = {}

    for file_path in directory:
        profile_data, profile_text = get_education_profile(file_path)

        # Check if profile_data is None
        if profile_data is None:
            print(f"Warning: No profile data extracted for {file_path}. Skipping...")
            continue

        # Try extracting contact info
        contact_info = profile_data.get("contact_information", {}).get("contact", {}).get("email")

        if contact_info:
            storage[contact_info] = profile_data
            storage_text[contact_info] = profile_text
        else:
            print(f"Warning: Missing contact information for {file_path}. Marking for further processing.")
            missing_contact_info[file_path] = profile_data
    
    return storage, storage_text, missing_contact_info

In [13]:
def fix_missing_contact_info(missing_entries):
    fixed_storage = {}

    for file_path, profile_data in missing_entries.items():
        # Attempt to extract from different fields
        contact_info = (
            profile_data.get("contact_information", {}).get("contact", {}).get("phone") or
            profile_data.get("contact_information", {}).get("contact", {}).get("linkedin") or
            profile_data.get("contact_information", {}).get("contact", {}).get("github")
        )

        if contact_info:
            print(f"Recovered contact info for {file_path} using alternative fields.")
            fixed_storage[contact_info] = profile_data
        else:
            print(f"Still missing contact info for {file_path}. Keeping it under its filename as key.")
            fixed_storage[file_path] = profile_data  # Use file path as fallback key

    return fixed_storage

In [14]:
resultant_metadata, resultant_metadata_text, missing_contact_info = aggMetadata(output_directory_list)
fixed_storage = fix_missing_contact_info(missing_entries=missing_contact_info)

Started parsing the file under job_id de665082-a7b2-48f8-b2b8-20adcee44597
.

BadRequestError: Error code: 400 - {'error': {'message': 'The model `mixtral-8x7b-32768` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}

In [ ]:
missing_contact_info

In [ ]:
fixed_storage

In [ ]:
resultant_metadata

In [ ]:
resultant_metadata_combined = {**resultant_metadata, **fixed_storage}

In [ ]:
resultant_metadata_combined

In [ ]:
import os
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [ ]:
from langchain_chroma import Chroma
vector_store_contact_information = Chroma(
    embedding_function=embeddings,
    collection_name="contact_information",
    persist_directory="./chroma_langchain_db_1",
)

from langchain_chroma import Chroma
vector_store_education = Chroma(
    embedding_function=embeddings,
    collection_name="education",
    persist_directory="./chroma_langchain_db_1",
)

from langchain_chroma import Chroma
vector_store_experience = Chroma(
    embedding_function=embeddings,
    collection_name="experience",
    persist_directory="./chroma_langchain_db_1",
)

from langchain_chroma import Chroma
vector_store_technical_skills = Chroma(
    embedding_function=embeddings,
    collection_name="technical_skills",
    persist_directory="./chroma_langchain_db_1",
)

from langchain_chroma import Chroma
vector_store_projects = Chroma(
    embedding_function=embeddings,
    collection_name="projects",
    persist_directory="./chroma_langchain_db_1",
)

from langchain_chroma import Chroma
vector_store_clubs_and_communities = Chroma(
    embedding_function=embeddings,
    collection_name="clubs_and_communities",
    persist_directory="./chroma_langchain_db_1",
)

from langchain_chroma import Chroma
vector_store_achievements= Chroma(
    embedding_function=embeddings,
    collection_name="achievements",
    persist_directory="./chroma_langchain_db_1",
)

from langchain_chroma import Chroma
vector_store_publications = Chroma(
    embedding_function=embeddings,
    collection_name="publications",
    persist_directory="./chroma_langchain_db_1",
)

In [ ]:
contact_information = {}
education = {}
experience = {}
technical_skills = {}
projects = {}
clubs_and_community = {}
achievements = {}
publications = {}

In [ ]:
def decoupling_component(resultant_metadata_combined):
    for i, k in resultant_metadata_combined.items():
        contact_information[i] = k.get("contact_information")
        education[i] = k.get("education")
        experience[i] = k.get("experience")
        technical_skills[i] = k.get("technical_skills")
        projects[i] = k.get("projects")
        clubs_and_community[i] = k.get("clubs_and_community")
        achievements[i] = k.get("achievements")
        publications[i] = k.get("publications")

In [ ]:
for i, k in contact_information.items():
    print(i)

In [ ]:
decoupling_component(resultant_metadata_combined=resultant_metadata_combined)

In [ ]:
education

In [ ]:
contact_information

In [ ]:
publications

In [ ]:
from uuid import uuid4
from langchain_core.documents import Document

def load_documents(json_file):
    docs = [Document(page_content=i + str(k), metadata={"identifier": i}) for i, k in json_file.items()]
    uuids_docs = [str(uuid4()) for _ in range(len(docs))]
    return docs, uuids_docs

In [ ]:
docs_contact_information, uuids_contact_information = load_documents(contact_information)
docs_education, uuids_education = load_documents(education)
docs_experience, uuids_experience = load_documents(experience)
docs_technical_skills, uuids_technical_skills = load_documents(technical_skills)
docs_projects, uuids_projects = load_documents(projects)
docs_clubs_and_community, uuids_clubs_and_community = load_documents(clubs_and_community)
docs_achievements, uuids_achievements = load_documents(achievements)
docs_publications, uuids_publications = load_documents(publications)

In [ ]:
docs_projects

In [ ]:
vector_store_contact_information.add_documents(documents=docs_contact_information, ids=uuids_contact_information)

In [ ]:
vector_store_education.add_documents(documents=docs_education, ids=uuids_education)

In [ ]:
vector_store_experience.add_documents(documents=docs_experience, ids=uuids_experience)

In [ ]:
vector_store_technical_skills.add_documents(documents=docs_technical_skills, ids=uuids_technical_skills)

In [ ]:
vector_store_projects.add_documents(documents=docs_projects, ids=uuids_projects)

In [ ]:
vector_store_clubs_and_communities.add_documents(documents=docs_clubs_and_community, ids=uuids_clubs_and_community)

In [ ]:
vector_store_achievements.add_documents(documents=docs_achievements, ids=uuids_achievements)

In [ ]:
vector_store_publications.add_documents(documents=docs_publications, ids=uuids_publications)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.chains import LLMChain
from langchain.schema.runnable import RunnableLambda

retriever = vector_store_projects.as_retriever(search_type="similarity")
retrieved_docs = retriever.invoke("Get me the email of a person names Neville")
# 
# context = "\n\n".join([doc.page_content for doc in retrieved_docs])
# 
# reasoning_prompt = ChatPromptTemplate.from_template(
#     "Given the following context:\n{context}\n\n"
#     "Decide how many documents to use to best answer the question."
# )
# 
# llm_chain = reasoning_prompt | llm
# final_docs = llm_chain.invoke({"context": context})
# retrieved_docs

In [ ]:
import datetime
from typing import Literal, Optional, Tuple
from langchain_core.pydantic_v1 import BaseModel, Field
class SubQuery(BaseModel):
    sub_query: str = Field(
        ...,
        description="A very specific query against the database.",
    )

In [ ]:
from langchain.output_parsers import PydanticToolsParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

system = """You are an expert at converting user questions into database queries.\ You have access to a database of tutorial videos about a software library for building LLM-powered applications.\ 
Perform query decomposition. Given a user question, break it down into distinct sub questions that \
you need to answer in order to answer the original question.
If there are acronyms or words you are not familiar with, do not try to rephrase them.
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

llm_openai = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0)
llm_with_tools = llm_openai.bind_tools([SubQuery])
parser = PydanticToolsParser(tools=[SubQuery])
query_analyser = prompt | llm_with_tools | parser

In [ ]:
subqueries = query_analyser.invoke({"Get me the details of the candidates who have done some projects in the LLMs or GenAi Space"})

In [ ]:
def extract_questions(subqueries_list):
    return [query_1.sub_query for query_1 in subqueries_list]

In [ ]:
list_of_sub_queries = extract_questions(subqueries)

In [ ]:
print(list_of_sub_queries)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import LLMChain
from langchain_core.output_parsers import StrOutputParser

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def retrieve_for_subqueries(subqueries, retriever):
    all_docs = []
    for query in subqueries:
        results = retriever.invoke(query)
        all_docs.extend(results)
    return all_docs

# Step 3: Reasoning Agent to Filter Retrieved Results
reasoning_prompt = ChatPromptTemplate.from_template(
    "You are an intelligent assistant. Given the original user question:\n"
    "{user_query}\n\n"
    "The question was decomposed into the following subqueries:\n"
    "{subqueries}\n\n"
    "Here are the retrieved results:\n\n"
    "{context}\n\n"
    "Based on the above, extract only the most relevant information and provide a final, concise answer, and format the answer properly in a json format before giving it out."
)

def reasoning_agent(user_query, subqueries, docs, llm):
    subquery_text = "\n".join([f"- {sq}" for sq in subqueries])
    context = "\n\n".join([doc.page_content for doc in docs])
    
    reasoning_chain = reasoning_prompt | llm | StrOutputParser()
    
    return reasoning_chain.invoke({
        "user_query": user_query,
        "subqueries": subquery_text,
        "context": context
    })

def retrieval_chain(user_query, retriever, llm):
    subqueries = list_of_sub_queries
    retrieved_docs = retrieve_for_subqueries(subqueries, retriever)
    final_answer = reasoning_agent(user_query, subqueries, retrieved_docs, llm)
    return final_answer

In [ ]:
query="Get me just the email of the candidates who have done some projects in the LLMs or Ge nAi Space"

In [ ]:
retrieval_chain(query, retriever, llm_openai)

In [ ]:
projects

In [ ]:

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.pydantic_v1 import BaseModel,Field
from langchain_openai import ChatOpenAI

class RouteQuery(BaseModel):
    datasource: Literal["vector_store_contact_information",
        "vector_store_education",
        "vector_store_experience",
        "vector_store_technical_skills",
        "vector_store_projects",
        "vector_store_clubs_and_community",
        "vector_store_achievements",
        "vector_store_publications",] = Field(
        ..., description="Given a user question choose which datasource would be most relevant for answering their question"
    )

In [ ]:
structured_llm = llm_openai.with_structured_output(RouteQuery)
system = """You are an expert at routing a user question to the appropriate data source.
Based on the programming language the question is referring to, route it to the relevant datasources"""

prompt = ChatPromptTemplate(
[    ("system", system),
    ("human", "{question}"),
]
)

router = prompt | structured_llm

In [ ]:
def route_subqueries(subqueries_list):
    result = {}
    for i in subqueries_list:
        data_source = router.invoke(i)
        result[i] = data_source
    return result 

In [ ]:
routing_mapping = route_subqueries(list_of_sub_queries)

In [ ]:
mappings_vector_stores = {"vector_store_contact_information":vector_store_contact_information,
 "vector_store_education":vector_store_education,
 "vector_store_experience":vector_store_experience,
 "vector_store_technical_skills":vector_store_technical_skills,
 "vector_store_projects":vector_store_projects,
 "vector_store_clubs_and_community":vector_store_clubs_and_communities,
 "vector_store_achievements":vector_store_achievements,
 "vector_store_publications":vector_store_publications}

In [ ]:
def retrieve_information_from_multiple_vector_stores(routing_map):
    for question, route in routing_map:
        retriever = mappings_vector_stores[route.datasource]
        query_1 = question
        result = retriever.invoke({"question": query_1})
        print(result)

In [ ]:
routing_mapping

In [ ]:
path_file = "C:\\Users\sreec\Downloads\Data_Scientist_Job_Description.pdf"

In [ ]:
import os
from dataclasses import dataclass
from pydantic_ai import Agent, RunContext, Tool
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.openai import OpenAIProvider

@dataclass
class MyDeps:
    file_path: str

system_prompt = """\
You are a Job Description Summarizer. Your task is to:
1. Extract key elements from job descriptions
2. Identify required skills, qualifications, and experience
3. Highlight main responsibilities
4. Present the summary in a clear, organized format
"""

def parse_jd(file_path: str) -> str:
    try:
        from llama_parse import LlamaParse
        parser = LlamaParse(
            parsing_instruction="Extract key job details including skills, experience, qualifications, and responsibilities",
            result_type="markdown"
        )
        parsed_data = parser.load_data(file_path)
        if not parsed_data or len(parsed_data) == 0:
            return "Error: No content could be parsed from the file"
        return "\n".join(parsed_data[0].text.split("\n"))    
    except Exception as e:
        return f"Error parsing file: {str(e)}"

def get_file_path(ctx: RunContext[MyDeps]) -> str:
    return ctx.deps.file_path

model = OpenAIModel(
    'gpt-3.5-turbo',
    provider=OpenAIProvider(api_key=os.environ["OPENAI_API_KEY"])
)

jd_agent = Agent(
    model=model,
    deps_type=MyDeps,
    tools=[
        Tool(parse_jd, takes_ctx=False),
        Tool(get_file_path, takes_ctx=True)
    ],
    system_prompt=system_prompt
)

if __name__ == '__main__':
    file_path = r"C:\Users\sreec\Downloads\Data_Scientist_Job_Description.pdf" 
    if not os.path.exists(file_path):
        print(f"Error: File not found at {file_path}")
    else:
        deps = MyDeps(file_path=file_path)
        result = jd_agent.run_sync(
            "Please summarize this job description", 
            deps=deps
        )
        print(result.data)

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
import ssl
from pydantic_ai import Agent, Tool
from typing import List
import json
import nest_asyncio
import asyncio

nest_asyncio.apply()

@dataclass
class EmailConfig:
    sender_email: str = "morningnamastelucifer@gmail.com"
    password: str = "nnki vnbm gnzp qyvj"  # Use env vars in production

@dataclass
class Candidate:
    name: str
    email: str
    position: str
    available_dates: List[str]

system_prompt = """\
You are an Interview Scheduling Assistant. Your tasks:
1. Generate personalized interview invitation emails
2. Propose suitable time slots
3. Include interview format details
4. Maintain professional tone
"""

def send_email(subject: str, body: str, to_email: str, config: EmailConfig) -> str:
    msg = MIMEMultipart()
    msg['From'] = config.sender_email
    msg['To'] = to_email
    msg['Subject'] = subject
    msg.attach(MIMEText(body, 'plain'))
    
    context = ssl.create_default_context()
    
    try:
        with smtplib.SMTP('smtp.gmail.com', 587) as server:
            server.starttls(context=context)
            server.login(config.sender_email, config.password)
            server.send_message(msg)
        return f"Email sent to {to_email}"
    except Exception as e:
        return f"Error: {str(e)}"

def generate_time_slots(start_date: str) -> List[str]:
    slots = []
    current_date = datetime.strptime(start_date, "%Y-%m-%d")
    
    for _ in range(3):  # Next 3 days
        for hour in [9, 11, 14, 16]:  # 4 slots per day
            slot = current_date.replace(hour=hour, minute=0)
            slots.append(slot.strftime("%A, %B %d at %I:%M %p"))
        current_date += timedelta(days=1)
    
    return slots

def create_email_content(candidate: Candidate, slots: List[str]) -> dict:
    return {
        "subject": f"Interview for {candidate.position}",
        "body": f"""Dear {candidate.name},
        
We're excited to interview you for {candidate.position}.

Available slots:
{chr(10).join(f"- {s}" for s in slots[:3])}

Please reply with your preferred time.

Best regards,
Hiring Team
"""
    }

# Create agent
interview_agent = Agent(
    'gpt-3.5-turbo',
    deps_type=EmailConfig,
    tools=[
        Tool(send_email, takes_ctx=False),
        Tool(generate_time_slots, takes_ctx=False),
        Tool(create_email_content, takes_ctx=False)
    ],
    system_prompt=system_prompt
)

def schedule_interviews(candidates: List[Candidate], start_date: str):
    """Run the scheduling process"""
    config = EmailConfig()
    
    for candidate in candidates:
        slots = generate_time_slots(start_date)
        email = create_email_content(candidate, slots)
        
        # Prepare context string
        context = json.dumps({
            "candidate": candidate.name,
            "position": candidate.position,
            "slots": slots[:3]
        })
        
        try:
            # Run agent synchronously with loop handling
            result = interview_agent.run_sync(
                f"Send interview email with these details: {context}",
                deps=config
            )
            print(result.data)
        except Exception as e:
            print(f"Error scheduling {candidate.name}: {str(e)}")

if __name__ == '__main__':
    candidates = [
        Candidate(
            name="John Doe",
            email="morningnamastelucifer@gmail.com",
            position="Data Scientist",
            available_dates=["2023-12-10"]
        )
    ]
    
    schedule_interviews(candidates, "2023-12-10")